In [1]:
from pathlib import Path
import polars as pl
from regex_generator import words_to_compact_regex

path = Path("/workspaces/example/data/projects/others/ricu/inst/extdata/config/concept-dict.json")
df = pl.read_json(path)
d_items_df = pl.scan_csv("/workspaces/example/data/mimic-iv-3.1/icu/d_items.csv.gz")
d_labitems_df = pl.scan_csv("/workspaces/example/data/mimic-iv-3.1/hosp/d_labitems.csv.gz")

In [2]:
entries = []
for column in df.columns:
    if (srcs := df[column][0].get("sources")) and (miiv := srcs.get("miiv")):
        parent_folder = df[column][0].get("category")
        name = df[column][0].get("description")
        extension_columns = {
            "dataset": " ",
            "table": " ",
        }
        if ids := miiv[0].get("ids"):
            if isinstance(ids, str) or (isinstance(ids, list) and isinstance(ids[0], str)):
                continue
            ids: list[str] = ids if isinstance(ids, list) else [ids]

            # TODO: get labels and write into regex
            filtered_items = d_items_df.filter(pl.col("itemid").is_in(ids)).select("label").collect()
            filtered_labitems = d_labitems_df.filter(pl.col("itemid").is_in(ids)).select("label").collect()

            regex = list(filtered_items.to_dict()["label"]) + list(filtered_labitems.to_dict()["label"])
        
        pattern = []
        for miiv_e in miiv:
            entry = {key:miiv_e.get(key) for key in miiv[0].keys()}
            if (not isinstance(ids, bool)) and (ids := entry.get("ids")) and (not entry.get("regex")):
                if isinstance(ids, str) or (isinstance(ids, list) and isinstance(ids[0], str)):
                    continue
                ids: list[str] = ids if isinstance(ids, list) else [ids]

                filtered_items = d_items_df.filter(pl.col("itemid").is_in(ids)).select("label").collect()
                filtered_labitems = d_labitems_df.filter(pl.col("itemid").is_in(ids)).select("label").collect()

                # regex = list(filtered_items.to_dict()["label"]) + list(filtered_labitems.to_dict()["label"])
                entry["regex"] = "|".join(list(filtered_items.to_dict()["label"]) + list(filtered_labitems.to_dict()["label"]))
                print(words_to_compact_regex(list(filtered_items.to_dict()["label"]) + list(filtered_labitems.to_dict()["label"])))

            if entry.get("regex"):
                entry["regex"] = "(" + entry["regex"] + ")"
            pattern.append(entry)

        mapping = {
            "dataset": "miiv",
            # "pattern":  [{key:miiv_e.get(key) for key in miiv[0].keys()} for miiv_e in miiv],
            "pattern":  pattern,
        }

        entries += {
            "parent_folder": parent_folder,
            "name": name,
            "extension_columns": extension_columns,
            "mapping":mapping,
        },       

new_entries: dict[str, list] = {}
for entry in entries:
    if not new_entries.get(entry["parent_folder"]):
        new_entries[entry["parent_folder"]] = []
    new_entries[entry["parent_folder"]].append(entry)
entries = new_entries

(Temperature Celsius)
((Skin Temperature|Temperature Fahrenheit))
(Norepinephrine)
(Creatinine)
(Sedimentation Rate)
(Dobutamine)
((Arterial Blood Pressure diastolic|Non Invasive Blood Pressure diastolic))
(Inspired O2 Fraction)
(Oxygen)
(Hematocrit)
(Potassium)
(pH)
(Basophils)
(Dextrose (10%|20%|50%))
(GCS \- Motor Response)
(Neutrophils)
(Epinephrine)
(Height)
(Lymphocytes)
(INR\(PT\))
(Lactate)
(Alkaline Phosphatase)
(MCHC)
(Dopamine)
(PTT)
(Phosphate)
((A(RT BP Mean|rterial Blood Pressure mean)|Non Invasive Blood Pressure mean))
(Creatine Kinase, MB Isoenzyme)
(Asparate Aminotransferase \(AST\))
((Invasive Ventilation|Non\-invasive Ventilation))
(Bands)
(Free Calcium)
(Eosinophils)
(MCV)
(PT)
(Bilirubin, Total)
(Red Blood Cells)
(Magnesium)
(Epinephrine)
(Hemoglobin)
(Glucose)
(Admission Weight \(Kg\))
(Respiratory Rate( \((Set\)|Total\)|spontaneous\))|))
(Alanine Aminotransferase \(ALT\))
(Heart Rate)
((Arterial O2 Saturation|O2 saturation pulseoxymetry|SpO2 Desat Limit))
(Oxygen

In [3]:
import json

with open("data.json", "w", encoding="utf-8") as f:
    json.dump(entries, f, indent=2, ensure_ascii=False)

In [4]:
for key in entries.keys():
    print(key)

vitals
medications
chemistry
hematology
outcome
blood gas
neurological
demographics
respiratory
microbiology
output


In [5]:
#  MIMIC.chemistry is subset of OpenICU.observation 
for entry in entries["chemistry"]:
    print(entry["name"])

creatinine
potassium
alkaline phosphatase
phosphate
creatine kinase MB
aspartate aminotransferase
total bilirubin
magnesium
glucose
alanine aminotransferase
chloride
troponin I
albumin
sodium
blood urea nitrogen
C-reactive protein
troponin t
calcium
creatine kinase
bicarbonate
bilirubin direct


In [6]:
d_labitems_df.filter(pl.col("label").str.contains(".*?(Urine Creatinine)")).head(10).collect()

itemid,label,fluid,category
i64,str,str,str
51106,"""Urine Creatinine""","""Urine""","""Chemistry"""


In [7]:
for entry in entries["chemistry"]:
    print(entry["name"], entry["mapping"]["pattern"][0]["regex"])

creatinine (Creatinine)
potassium (Potassium)
alkaline phosphatase (Alkaline Phosphatase)
phosphate (Phosphate)
creatine kinase MB (Creatine Kinase, MB Isoenzyme)
aspartate aminotransferase (Asparate Aminotransferase (AST))
total bilirubin (Bilirubin, Total)
magnesium (Magnesium)
glucose (Glucose|Glucose)
alanine aminotransferase (Alanine Aminotransferase (ALT))
chloride (Chloride)
troponin I (Troponin I)
albumin (Albumin)
sodium (Sodium)
blood urea nitrogen (Urea Nitrogen)
C-reactive protein (C-Reactive Protein)
troponin t (Troponin T)
calcium (Calcium, Total)
creatine kinase (Creatine Kinase (CK))
bicarbonate (Bicarbonate)
bilirubin direct (Bilirubin, Direct)


In [8]:
for key in entries.keys():
    print(key)
    for entry in entries[key]:
        print(entry)

vitals
{'parent_folder': 'vitals', 'name': 'temperature', 'extension_columns': {'dataset': ' ', 'table': ' '}, 'mapping': {'dataset': 'miiv', 'pattern': [{'ids': [223762], 'table': 'chartevents', 'sub_var': 'itemid', 'callback': None, 'regex': '(Temperature Celsius)'}, {'ids': [223761, 224027], 'table': 'chartevents', 'sub_var': 'itemid', 'callback': "convert_unit(fahr_to_cels, 'C', 'f')", 'regex': '(Temperature Fahrenheit|Skin Temperature)'}]}}
{'parent_folder': 'vitals', 'name': 'diastolic blood pressure', 'extension_columns': {'dataset': ' ', 'table': ' '}, 'mapping': {'dataset': 'miiv', 'pattern': [{'ids': [220051, 220180], 'table': 'chartevents', 'sub_var': 'itemid', 'regex': '(Arterial Blood Pressure diastolic|Non Invasive Blood Pressure diastolic)'}]}}
{'parent_folder': 'vitals', 'name': 'mean arterial pressure', 'extension_columns': {'dataset': ' ', 'table': ' '}, 'mapping': {'dataset': 'miiv', 'pattern': [{'ids': [220052, 220181, 225312], 'table': 'chartevents', 'sub_var': 'it